In [43]:
# 1. Simulating the JSON output we got from the AI
agent_decision = {
    'category': 'Refund', 
    'order_id': '1234', 
    'sentiment': 'Positive', 
    'urgency': 2
}

print(f"Processing email for Order {agent_decision['order_id']}...")

# 2. The Decision Logic (The "Brain")
if agent_decision['category'] == 'Refund':
    
    # Check if the customer is angry or it's highly urgent
    if agent_decision['urgency'] >= 4 or agent_decision['sentiment'] == 'Negative':
        print("🚨 ACTION: Escalate to human manager. Customer is upset or urgency is high.")
        print("Routing to: Human Support Queue")
    else:
        print("✅ ACTION: Auto-approve refund.")
        print("Routing to: Payment API (Simulated)")

elif agent_decision['category'] == 'Order Status':
    print("🔍 ACTION: Query database for tracking number.")
    
else:
    print("❓ ACTION: Route to general queue.")

Processing email for Order 1234...
✅ ACTION: Auto-approve refund.
Routing to: Payment API (Simulated)


In [44]:
# 1. Define the Tool (The Action the agent can take)
def process_refund_api(order_id):
    # In the real world, this would talk to Stripe or PayPal.
    # For now, it simulates a successful API call.
    return f"Refund successfully processed for Order #{order_id}. Money returned to card."

# 2. The Agent's Decision Data (From the previous step)
agent_decision = {
    'category': 'Refund', 
    'order_id': '1234', 
    'sentiment': 'Positive', 
    'urgency': 2
}

print(f"Processing email for Order {agent_decision['order_id']}...")

# 3. The Decision Engine + Action
if agent_decision['category'] == 'Refund':
    
    if agent_decision['urgency'] >= 4 or agent_decision['sentiment'] == 'Negative':
        print("🚨 ACTION: Escalate to human manager. Customer is upset or urgency is high.")
    else:
        print("✅ ACTION: Auto-approve refund. Executing tool...")
        
        # --- THIS IS THE MAGIC LINE ---
        # We actually CALL the tool function and pass the order_id to it
        result = process_refund_api(agent_decision['order_id'])
        
        # Print the result returned by the tool
        print(f"Tool Output: {result}")

Processing email for Order 1234...
✅ ACTION: Auto-approve refund. Executing tool...
Tool Output: Refund successfully processed for Order #1234. Money returned to card.


In [45]:
import json
from groq import Groq
import os
from dotenv import load_dotenv

# 1. Load API Key
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# 2. Define the actual Python functions (The "Hands")
def process_refund_api(order_id):
    return f"Refund successfully processed for Order #{order_id}."

def check_inventory(product_name):
    # Simulated database lookup
    return f"Inventory for {product_name}: 45 units in stock."

# 3. Describe the tools to the AI (The "Menu")
# The AI reads these descriptions to decide which tool to use.
tools = [
    {
        "type": "function",
        "function": {
            "name": "process_refund_api",
            "description": "Process a refund for a customer order. Use this when the customer asks for their money back.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID, e.g., 1234"}
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Check how many coffee beans are in stock. Use this when a customer asks about availability.",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string", "description": "The name of the coffee product"}
                },
                "required": ["product_name"]
            }
        }
    }
]

# 4. The Customer Message
customer_message = "Hi, I want to return my order #1234 and get my money back."

print(f"Customer: {customer_message}")

# 5. Ask the AI to choose a tool
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": customer_message}],
    tools=tools,
    tool_choice="auto" # Tells the AI: "You decide which tool to use."
)

# 6. See what the AI decided
tool_call = response.choices[0].message.tool_calls[0]
print(f"\nAI Decision: I need to use the '{tool_call.function.name}' tool.")
print(f"With arguments: {tool_call.function.arguments}")

Customer: Hi, I want to return my order #1234 and get my money back.

AI Decision: I need to use the 'process_refund_api' tool.
With arguments: {"order_id":"1234"}


In [47]:
# 1. We have the tool_call from the AI (from the previous cell)
tool_name = tool_call.function.name
tool_args = json.loads(tool_call.function.arguments)

print(f"Executing tool: {tool_name}")

# 2. Map the tool name to the actual Python function
if tool_name == "process_refund_api":
    # Call the function we defined earlier
    result = process_refund_api(tool_args["order_id"])
elif tool_name == "check_inventory":
    result = check_inventory(tool_args["product_name"])
else:
    result = "Unknown tool."

print(f"Tool Result: {result}")

# 3. (Optional) Send the result back to the AI so it can write a nice reply to the customer
final_response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "user", "content": customer_message},
        {"role": "assistant", "tool_calls": [tool_call]},
        {"role": "tool", "tool_call_id": tool_call.id, "content": result}
    ]
)

print(f"\nFinal message to customer: {final_response.choices[0].message.content}")

Executing tool: process_refund_api
Tool Result: Refund successfully processed for Order #1234.

Final message to customer: Your refund for order **#1234** has been processed successfully. The full amount will be credited back to the original payment method within 3–5 business days, depending on your bank.

If you have any other questions or need further assistance, just let me know!
